## IMPORTS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import sys
import duckdb
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Import shared utilities
sys.path.insert(0, os.path.abspath('.'))
from notebook_utils import (
    drop_high_missing_columns, impute_features,
    plot_trajectory_distribution, plot_boxplots_with_stats,
    plot_roc_pr_curves, print_statistical_comparisons, train_repeated_cv, biomarker_summary_stats
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports successful")
print("✓ Shared utilities loaded")

✓ Imports successful
✓ Shared utilities loaded


## LOAD DATA

In [2]:
# Connect to HiRiD DuckDB
db_path = '/home/gaga/data/physionet/HiRiD/hirid.duckdb'
conn = duckdb.connect(db_path, read_only=True)

print(f"✓ Connected to HiRiD database: {db_path}")

✓ Connected to HiRiD database: /home/gaga/data/physionet/HiRiD/hirid.duckdb


In [3]:
# Load patient cohort - sepsis patients
# Define sepsis as: infection + organ dysfunction (SOFA-like)
# Infection: elevated CRP or procalcitonin or WBC

n_sample_patients = 20000

# Step 1: Get patient IDs with signs of infection
infection_query = f"""
SELECT DISTINCT CAST(o.patientid AS INTEGER) as patientid
FROM observations o
WHERE 
    (
        -- CRP > 100 mg/L
        (o.variableid = '20002200' AND CAST(o.value AS DOUBLE) > 100)
        -- Procalcitonin > 0.5 ug/L
        OR (o.variableid = '24000570' AND CAST(o.value AS DOUBLE) > 0.5)
        -- WBC < 4 or > 12 G/L
        OR (o.variableid = '20000700' AND (CAST(o.value AS DOUBLE) < 4 OR CAST(o.value AS DOUBLE) > 12))
        -- Temperature < 36 or > 38.3
        OR (o.variableid = '410' AND (CAST(o.value AS DOUBLE) < 36 OR CAST(o.value AS DOUBLE) > 38.3))
    )
LIMIT {n_sample_patients}
"""

infection_patients = conn.execute(infection_query).fetchdf()['patientid'].tolist()
print(f"Step 1: Found {len(infection_patients):,} patients with infection signs")

# Step 2: Get patient demographics for those with infection
patient_query = f"""
SELECT 
    CAST(g.patientid AS INTEGER) as patientid,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time,
    g.sex,
    CAST(g.age AS INTEGER) as age,
    g.discharge_status,
    COUNT(DISTINCT o.datetime) as n_observations,
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 as los_days
FROM ref_general_table g
INNER JOIN observations o ON g.patientid = o.patientid
WHERE CAST(g.patientid AS INTEGER) IN {tuple(infection_patients)}
GROUP BY g.patientid, g.admissiontime, g.sex, g.age, g.discharge_status
HAVING 
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 >= 2.0
    AND COUNT(DISTINCT o.datetime) >= 100
"""

patient_df = conn.execute(patient_query).fetchdf()

print(f"\n✓ Loaded {len(patient_df):,} sepsis patients")
print(f"  Mean LOS: {patient_df['los_days'].mean():.1f} days")
print(f"  Mean age: {patient_df['age'].mean():.1f} years")
print(f"  Sex distribution: {patient_df['sex'].value_counts().to_dict()}")

Step 1: Found 19,631 patients with infection signs

✓ Loaded 7,497 sepsis patients
  Mean LOS: 6.5 days
  Mean age: 62.5 years
  Sex distribution: {'M': 4927, 'F': 2570}


In [4]:
# Sample subset for analysis
n_patients = min(5000, len(patient_df))
patient_subset = patient_df.sample(n_patients, random_state=920)['patientid'].tolist()

print(f"✓ Using {len(patient_subset):,} patients for analysis")

✓ Using 5,000 patients for analysis


In [5]:
# Load lactate measurements (primary biomarker for shock)
# Lactate IDs: 24000524 (arterial), 24000732, 24000485 (venous)
lactate_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as lactate,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid IN ('24000524', '24000732', '24000485')
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0
    AND CAST(o.value AS DOUBLE) < 30  -- Remove outliers
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading lactate measurements...")
lactate_df = conn.execute(lactate_query).fetchdf()

print(f"\n✓ Loaded {len(lactate_df):,} lactate measurements")
print(f"  Patients with lactate: {lactate_df['patientid'].nunique():,}")
if len(lactate_df) > 0:
    print(f"  Mean lactate: {lactate_df['lactate'].mean():.2f} mmol/L")
    print(f"  Median lactate: {lactate_df['lactate'].median():.2f} mmol/L")

Loading lactate measurements...

✓ Loaded 126,490 lactate measurements
  Patients with lactate: 4,851
  Mean lactate: 1.94 mmol/L
  Median lactate: 1.40 mmol/L


In [6]:
# Load platelet and WBC measurements (additional biomarkers)
# Platelets ID: 20000110 (G/L)
# Leukocytes (WBC) ID: 20000700 (G/L)
platelet_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as platelets,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '20000110'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0
    AND CAST(o.value AS DOUBLE) < 1500  -- Remove outliers (G/L)
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

wbc_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as wbc,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '20000700'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0.1
    AND CAST(o.value AS DOUBLE) < 100  -- Remove outliers (G/L)
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading platelet and WBC measurements...")
platelet_df = conn.execute(platelet_query).fetchdf()
wbc_df = conn.execute(wbc_query).fetchdf()

print(f"\n✓ Loaded {len(platelet_df):,} platelet measurements")
print(f"  Patients with platelets: {platelet_df['patientid'].nunique():,}")
print(f"✓ Loaded {len(wbc_df):,} WBC measurements")
print(f"  Patients with WBC: {wbc_df['patientid'].nunique():,}")
if len(platelet_df) > 0:
    print(f"  Median platelets: {platelet_df['platelets'].median():.1f} G/L")
if len(wbc_df) > 0:
    print(f"  Median WBC: {wbc_df['wbc'].median():.1f} G/L")

Loading platelet and WBC measurements...

✓ Loaded 88,425 platelet measurements
  Patients with platelets: 4,991
✓ Loaded 89,935 WBC measurements
  Patients with WBC: 4,984
  Median platelets: 170.0 G/L
  Median WBC: 11.4 G/L


In [7]:
# Load vasopressor administration (pharma table)
vasopressor_query = f"""
SELECT 
    CAST(p.patientid AS INTEGER) as patientid,
    CAST(p.givenat AS TIMESTAMP) as charttime,
    p.pharmaid,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM pharma_records p
INNER JOIN ref_general_table g ON p.patientid = g.patientid
WHERE 
    -- Noradrenalin
    p.pharmaid IN ('1000462', '1000656', '1000657', '1000658')
    -- Adrenalin
    OR p.pharmaid IN ('71', '1000750', '1000649', '1000650', '1000655')
    -- Vasopressin
    OR p.pharmaid IN ('112', '113')
    -- Dobutamine
    OR p.pharmaid = '426'
    AND CAST(p.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY p.patientid, p.givenat
"""

print("Loading vasopressor data...")
vasopressor_df = conn.execute(vasopressor_query).fetchdf()

print(f"\n✓ Loaded {len(vasopressor_df):,} vasopressor records")
print(f"  Patients on vasopressors: {vasopressor_df['patientid'].nunique():,}")

Loading vasopressor data...

✓ Loaded 1,737,440 vasopressor records
  Patients on vasopressors: 12,252


In [8]:
# Load vitals and labs - identify available variables from reference table
# Based on HiRiD variable reference, get IDs for key clinical variables

key_vars_query = """
SELECT DISTINCT
    CAST("ID" AS INTEGER) as variableid,
    "Variable Name" as variable_name,
    "Unit" as unit,
    "Source Table" as source_table
FROM ref_hirid_variable_reference
WHERE 
    -- Vitals
    "Variable Name" = 'Heart rate'
    OR "Variable Name" = 'Core body temperature'
    OR "Variable Name" = 'Invasive systolic arterial pressure'
    OR "Variable Name" = 'Invasive diastolic arterial pressure'
    OR "Variable Name" = 'Invasive mean arterial pressure'
    OR "Variable Name" = 'Non-invasive systolic arterial pressure'
    OR "Variable Name" = 'Non-invasive diastolic arterial pressure'
    OR "Variable Name" = 'Non-invasive mean arterial pressure'
    OR "Variable Name" = 'Respiratory rate'
    OR "Variable Name" = 'Peripheral oxygen saturation'
    -- Labs - Blood Chemistry
    OR "Variable Name" = 'Glucose [Moles/volume] in Serum or Plasma'
    OR "Variable Name" = 'Sodium [Moles/volume] in Blood'
    OR "Variable Name" = 'Potassium [Moles/volume] in Blood'
    OR "Variable Name" = 'Chloride [Moles/volume] in Blood'
    OR "Variable Name" = 'Bicarbonate [Moles/volume] in Arterial blood'
    OR "Variable Name" = 'Calcium.ionized [Moles/volume] in Blood'
    OR "Variable Name" = 'Magnesium [Moles/volume] in Blood'
    OR "Variable Name" = 'Phosphate [Moles/volume] in Blood'
    -- Labs - Renal
    OR "Variable Name" = 'Urea [Moles/volume] in Venous blood'
    OR "Variable Name" = 'Creatinine [Moles/volume] in Blood'
    -- Labs - Hematology
    OR "Variable Name" = 'Hemoglobin [Mass/volume] in Blood'
    OR "Variable Name" = 'Leukocytes [#/volume] in Blood'
    OR "Variable Name" = 'Platelets [#/volume] in Blood'
    -- Labs - Coagulation
    OR "Variable Name" = 'INR in Blood by Coagulation assay'
    OR "Variable Name" = 'aPTT in Blood by Coagulation assay'
    -- Labs - Liver/Metabolic
    OR "Variable Name" = 'Bilirubin.total [Moles/volume] in Serum or Plasma'
    OR "Variable Name" = 'Albumin [Mass/volume] in Serum or Plasma'
    -- Labs - Other
    OR "Variable Name" = 'Lactate [Mass/volume] in Arterial blood'
    -- Body measurements
    OR "Variable Name" = 'Body weight'
ORDER BY "Variable Name"
"""

key_vars = conn.execute(key_vars_query).fetchdf()
key_vars['unit'] = key_vars['unit'].fillna('N/A')
print(f"\n📊 Key clinical variables available in HiRiD:")
print(f"{'Variable ID':<12} {'Variable Name':<60} {'Unit':<15}")
print("="*100)
for _, row in key_vars.iterrows():
    print(f"{row['variableid']:<12} {row['variable_name']:<60} {row['unit']:<15}")

print(f"\n✓ Found {len(key_vars)} key clinical variables")

# Create variable ID to column name mapping (handle BP variants properly)
var_id_to_col = {}
for _, row in key_vars.iterrows():
    var_name = row['variable_name']
    var_id = str(row['variableid'])
    
    # Create concise column names, keeping BP types separate
    if 'Invasive systolic' in var_name:
        col_name = 'sbp_invasive'
    elif 'Invasive diastolic' in var_name:
        col_name = 'dbp_invasive'
    elif 'Invasive mean arterial' in var_name:
        col_name = 'mbp_invasive'
    elif 'Non-invasive systolic' in var_name:
        col_name = 'sbp_noninvasive'
    elif 'Non-invasive diastolic' in var_name:
        col_name = 'dbp_noninvasive'
    elif 'Non-invasive mean arterial' in var_name:
        col_name = 'mbp_noninvasive'
    elif 'Heart rate' in var_name:
        col_name = 'heart_rate'
    elif 'temperature' in var_name.lower():
        col_name = 'temperature'
    elif 'Respiratory rate' in var_name:
        col_name = 'respiratory_rate'
    elif 'oxygen saturation' in var_name.lower():
        col_name = 'spo2'
    elif 'Glucose' in var_name:
        col_name = 'glucose'
    elif 'Sodium' in var_name:
        col_name = 'sodium'
    elif 'Potassium' in var_name:
        col_name = 'potassium'
    elif 'Chloride' in var_name:
        col_name = 'chloride'
    elif 'Bicarbonate' in var_name:
        col_name = 'bicarbonate'
    elif 'Calcium' in var_name:
        col_name = 'calcium'
    elif 'Magnesium' in var_name:
        col_name = 'magnesium'
    elif 'Phosphate' in var_name:
        col_name = 'phosphate'
    elif 'Urea' in var_name:
        col_name = 'bun'
    elif 'Creatinine' in var_name:
        col_name = 'creatinine'
    elif 'Hemoglobin' in var_name:
        col_name = 'hemoglobin'
    elif 'Leukocytes' in var_name:
        col_name = 'wbc'
    elif 'Platelets' in var_name:
        col_name = 'platelets'
    elif 'INR' in var_name:
        col_name = 'inr'
    elif 'aPTT' in var_name:
        col_name = 'ptt'
    elif 'Bilirubin' in var_name:
        col_name = 'bilirubin'
    elif 'Albumin' in var_name:
        col_name = 'albumin'
    elif 'Lactate' in var_name:
        col_name = 'lactate'
    elif 'weight' in var_name.lower():
        col_name = 'weight'
    else:
        col_name = var_name.lower().replace(' ', '_')[:30]
    
    var_id_to_col[var_id] = col_name

print(f"\n📋 Variable mapping (ID → Column name):")
for var_id, col_name in sorted(var_id_to_col.items(), key=lambda x: x[1]):
    var_name = key_vars[key_vars['variableid'] == int(var_id)]['variable_name'].iloc[0]
    print(f"   {var_id:>10} → {col_name:25s} ({var_name})")

# Load observations for these variables from the patient subset
var_ids_str = "', '".join(var_id_to_col.keys())
obs_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    o.variableid,
    CAST(o.value AS DOUBLE) as value
FROM observations o
WHERE 
    o.variableid IN ('{var_ids_str}')
    AND o.value IS NOT NULL
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print(f"\n⏳ Loading observations for {len(patient_subset):,} patients...")
obs_df = conn.execute(obs_query).fetchdf()

# Map variable IDs to column names
obs_df['variable'] = obs_df['variableid'].astype(str).map(var_id_to_col)

print(f"\n✓ Loaded {len(obs_df):,} observations")
print(f"  Unique patients: {obs_df['patientid'].nunique():,}")
print(f"  Unique variables: {obs_df['variable'].nunique()}")
print(f"  Time range: {obs_df['charttime'].min()} to {obs_df['charttime'].max()}")

# Show observation counts per variable
print(f"\n📊 Observations per variable:")
var_counts = obs_df.groupby('variable').size().sort_values(ascending=False)
for var_name, count in var_counts.items():
    print(f"   {var_name:25s}: {count:>10,} observations")


📊 Key clinical variables available in HiRiD:
Variable ID  Variable Name                                                Unit           
24000605     Albumin [Mass/volume] in Serum or Plasma                     g/L            
20004200     Bicarbonate [Moles/volume] in Arterial blood                 mmol/l         
20004300     Bilirubin.total [Moles/volume] in Serum or Plasma            umol/l         
10000400     Body weight                                                  kg             
24000522     Calcium.ionized [Moles/volume] in Blood                      mmol/l         
24000439     Chloride [Moles/volume] in Blood                             mmol/l         
24000521     Chloride [Moles/volume] in Blood                             mmol/l         
410          Core body temperature                                        °C             
20000600     Creatinine [Moles/volume] in Blood                           umol/l         
24000523     Glucose [Moles/volume] in Serum or Plasma

In [9]:
# convert creatinine in vitals_labs to mg/dl
obs_df.loc[obs_df['variable'] == 'creatinine', 'value'] = obs_df.loc[obs_df['variable'] == 'creatinine', 'value'] / 88.4

## PREPROCESS

In [10]:
# Calculate baseline lactate (first 24h)
lactate_df['charttime'] = pd.to_datetime(lactate_df['charttime'])
lactate_df['admission_time'] = pd.to_datetime(lactate_df['admission_time'])

first_24h = lactate_df[lactate_df['charttime'] <= lactate_df['admission_time'] + pd.Timedelta(hours=24)]

baseline_lactate = first_24h.groupby('patientid')['lactate'].first().reset_index()
baseline_lactate.columns = ['patientid', 'baseline_lactate']

print(f"\nBaseline Lactate:")
print(f"   Patients with baseline: {len(baseline_lactate):,}")
print(f"   Mean: {baseline_lactate['baseline_lactate'].mean():.2f} mmol/L")
print(f"   Median: {baseline_lactate['baseline_lactate'].median():.2f} mmol/L")


Baseline Lactate:
   Patients with baseline: 4,701
   Mean: 2.60 mmol/L
   Median: 1.70 mmol/L


In [11]:
# Calculate baselines for platelets and WBC (first 24h)
platelet_df['charttime'] = pd.to_datetime(platelet_df['charttime'])
platelet_df['admission_time'] = pd.to_datetime(platelet_df['admission_time'])
wbc_df['charttime'] = pd.to_datetime(wbc_df['charttime'])
wbc_df['admission_time'] = pd.to_datetime(wbc_df['admission_time'])

platelet_first_24h = platelet_df[platelet_df['charttime'] <= platelet_df['admission_time'] + pd.Timedelta(hours=24)]
wbc_first_24h = wbc_df[wbc_df['charttime'] <= wbc_df['admission_time'] + pd.Timedelta(hours=24)]

baseline_platelets = platelet_first_24h.groupby('patientid')['platelets'].first().reset_index()
baseline_platelets.columns = ['patientid', 'baseline_platelets']

baseline_wbc = wbc_first_24h.groupby('patientid')['wbc'].first().reset_index()
baseline_wbc.columns = ['patientid', 'baseline_wbc']

print("\nBaseline Platelets / WBC:")
print(f"   Patients with platelet baseline: {len(baseline_platelets):,}")
print(f"   Median platelets: {baseline_platelets['baseline_platelets'].median():.1f} G/L")
print(f"   Patients with WBC baseline: {len(baseline_wbc):,}")
print(f"   Median WBC: {baseline_wbc['baseline_wbc'].median():.1f} G/L")


Baseline Platelets / WBC:
   Patients with platelet baseline: 4,786
   Median platelets: 174.0 G/L
   Patients with WBC baseline: 4,783
   Median WBC: 11.3 G/L


In [12]:
# Filter patients with sufficient measurements
lactate_counts = lactate_df.groupby('patientid').size()
valid_patients = lactate_counts[lactate_counts >= 3].index

lactate_filtered = lactate_df[lactate_df['patientid'].isin(valid_patients)]
patient_final = patient_df[patient_df['patientid'].isin(valid_patients)].merge(
    baseline_lactate, on='patientid', how='inner'
).merge(
    baseline_platelets, on='patientid', how='left'
).merge(
    baseline_wbc, on='patientid', how='left'
)

print(f"\nFiltering:")
print(f"   Patients with ≥3 lactate measurements: {len(patient_final):,}")


Filtering:
   Patients with ≥3 lactate measurements: 4,638


In [13]:
# Align platelets and WBC to valid sepsis patients and build daily values
valid_ids = set(patient_final['patientid'])

platelet_filtered = platelet_df[platelet_df['patientid'].isin(valid_ids)]
wbc_filtered = wbc_df[wbc_df['patientid'].isin(valid_ids)]

# Merge baselines
platelet_filtered = platelet_filtered.merge(baseline_platelets, on='patientid', how='left')
wbc_filtered = wbc_filtered.merge(baseline_wbc, on='patientid', how='left')

# Build time variables
platelet_filtered['time_days'] = (platelet_filtered['charttime'] - platelet_filtered['admission_time']).dt.total_seconds() / 86400
platelet_filtered['time_day'] = platelet_filtered['time_days'].astype(int)

wbc_filtered['time_days'] = (wbc_filtered['charttime'] - wbc_filtered['admission_time']).dt.total_seconds() / 86400
wbc_filtered['time_day'] = wbc_filtered['time_days'].astype(int)


print("\n✓ Platelet/WBC daily values prepared")


✓ Platelet/WBC daily values prepared


In [14]:
# Create time series
lactate_ts = lactate_filtered.merge(
    patient_final[['patientid', 'baseline_lactate', 'admission_time']], 
    on=['patientid', 'admission_time'], 
    how='left'
)

lactate_ts['time_days'] = (lactate_ts['charttime'] - lactate_ts['admission_time']).dt.total_seconds() / 86400
lactate_ts['time_day'] = lactate_ts['time_days'].astype(int)

# Rename for trajectory script
lactate_ts = lactate_ts.rename(columns={'lactate': 'lab_value'})

print(f"\n✓ Time series created")
print(f"   Total measurements: {len(lactate_ts):,}")
print(f"   Time range: {lactate_ts['time_days'].min():.1f} to {lactate_ts['time_days'].max():.1f} days")


✓ Time series created
   Total measurements: 126,356
   Time range: -0.0 to 28.0 days


In [15]:
# Create platelets time series
platelet_ts = platelet_filtered.merge(
    patient_final[['patientid', 'baseline_platelets', 'admission_time']], 
    on=['patientid', 'admission_time'], 
    how='left'
)

platelet_ts['time_days'] = (platelet_ts['charttime'] - platelet_ts['admission_time']).dt.total_seconds() / 86400
platelet_ts['time_day'] = platelet_ts['time_days'].astype(int)

# Rename for trajectory script
platelet_ts = platelet_ts.rename(columns={'platelets': 'lab_value'})
print(f"\n✓ Time series created")
print(f"   Total measurements: {len(platelet_ts):,}")
print(f"   Time range: {platelet_ts['time_days'].min():.1f} to {platelet_ts['time_days'].max():.1f} days")

# Create wbc time series
wbc_ts = wbc_filtered.merge(
    patient_final[['patientid', 'baseline_wbc', 'admission_time']], 
    on=['patientid', 'admission_time'],
    how='left'
)
wbc_ts['time_days'] = (wbc_ts['charttime'] - wbc_ts['admission_time']).dt.total_seconds() / 86400
wbc_ts['time_day'] = wbc_ts['time_days'].astype(int)
# Rename for trajectory script
wbc_ts = wbc_ts.rename(columns={'wbc': 'lab_value'})
print(f"\n✓ Time series created")
print(f"   Total measurements: {len(wbc_ts):,}")
print(f"   Time range: {wbc_ts['time_days'].min():.1f} to {wbc_ts['time_days'].max():.1f} days")


✓ Time series created
   Total measurements: 84,320
   Time range: -0.0 to 28.0 days

✓ Time series created
   Total measurements: 85,647
   Time range: -0.0 to 28.0 days


In [16]:
lactate_ts = lactate_ts.drop_duplicates(subset=['patientid', 'charttime'])
platelet_ts = platelet_ts.drop_duplicates(subset=['patientid', 'charttime'])
wbc_ts = wbc_ts.drop_duplicates(subset=['patientid', 'charttime'])

In [17]:
# Save lactate time series for trajectory computation
output_path = '../../results/hirid/sepsis/lactate_timeseries.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

lactate_ts.to_csv(output_path, index=False)
platelet_ts.to_csv('../../results/hirid/sepsis/platelet_timeseries.csv', index=False)
wbc_ts.to_csv('../../results/hirid/sepsis/wbc_timeseries.csv', index=False)

print(f"\n✓ Saved lactate time series: {output_path}")
print(f"✓ Saved platelet time series: ../../results/hirid/sepsis/platelet_timeseries.csv")
print(f"✓ Saved WBC time series: ../../results/hirid/sepsis/wbc_timeseries.csv")
print(f"   Shape: {lactate_ts.shape}")


✓ Saved lactate time series: ../../results/hirid/sepsis/lactate_timeseries.csv
✓ Saved platelet time series: ../../results/hirid/sepsis/platelet_timeseries.csv
✓ Saved WBC time series: ../../results/hirid/sepsis/wbc_timeseries.csv
   Shape: (93775, 7)


In [18]:
# Merge observations with patient admission times to calculate time_day
obs_df = obs_df.merge(
    patient_final[['patientid', 'admission_time']],
    on='patientid',
    how='inner'
)

obs_df['charttime'] = pd.to_datetime(obs_df['charttime'])
obs_df['admission_time'] = pd.to_datetime(obs_df['admission_time'])
obs_df['time_day'] = ((obs_df['charttime'] - obs_df['admission_time']).dt.total_seconds() / 86400).astype(int)

print(f"\n✓ Calculated time_day for observations")
print(f"  Time range: {obs_df['time_day'].min()} to {obs_df['time_day'].max()} days")

# Define valid ranges for each variable (clinically plausible values)
valid_ranges = {
    # Vitals
    'heart_rate': (20, 250),
    'temperature': (32, 42),
    'sbp_invasive': (40, 250),
    'dbp_invasive': (20, 180),
    'mbp_invasive': (30, 200),
    'sbp_noninvasive': (40, 250),
    'dbp_noninvasive': (20, 180),
    'mbp_noninvasive': (30, 200),
    'respiratory_rate': (4, 60),
    'spo2': (50, 100),
    
    # Labs - Chemistry
    'glucose': (20, 800),      # mg/dL (needs conversion from mmol/L)
    'sodium': (100, 180),      # mmol/L
    'potassium': (1.5, 10),    # mmol/L
    'chloride': (60, 140),     # mmol/L
    'bicarbonate': (5, 50),    # mmol/L
    'calcium': (0.5, 2.0),     # mmol/L (ionized)
    'magnesium': (0.3, 3.0),   # mmol/L
    'phosphate': (0.3, 4.0),   # mmol/L
    
    # Labs - Renal
    'bun': (0.5, 50),          # mmol/L (urea)
    'creatinine': (0.1, 15),  # mg/dL 
    
    # Labs - Hematology
    'hemoglobin': (40, 200),   # g/L
    'wbc': (0.1, 100),         # G/L (10^9/L)
    'platelets': (5, 1500),    # G/L (10^9/L)
    
    # Labs - Coagulation
    'inr': (0.5, 15),
    'ptt': (10, 300),          # seconds (aPTT)
    
    # Labs - Liver
    'bilirubin': (2, 800),     # µmol/L
    'albumin': (10, 60),       # g/L
    
    # Labs - Other
    'lactate': (0.2, 30),      # mmol/L
    'weight': (30, 300),       # kg
}

valid_ranges_df = pd.DataFrame.from_dict(valid_ranges, orient='index', columns=['min', 'max'])

print(f"\n📋 Defined valid ranges for {len(valid_ranges)} variables")

# Filter observations based on valid ranges
obs_df = obs_df.merge(
    valid_ranges_df.reset_index().rename(columns={'index': 'variable'}),
    on='variable',
    how='left'
)
obs_clean = obs_df[obs_df['value'].between(obs_df['min'], obs_df['max'])]

print(f"\n✓ Filtered observations")
print(f"  Before: {len(obs_df):,}")
print(f"  After:  {len(obs_clean):,}")
print(f"  Removed: {len(obs_df) - len(obs_clean):,} ({100*(len(obs_df)-len(obs_clean))/len(obs_df):.1f}%)")

# Aggregate vitals and labs by patient and time_day
# Calculate min, max, mean, std for each variable
print(f"\n⏳ Aggregating observations...")

vitals_labs_agg = obs_clean.pivot_table(
    index=['patientid', 'time_day'],
    columns='variable',
    values='value',
    aggfunc=['min', 'max', 'mean']
)

vitals_labs_agg.columns = [f"{var}_{stat}" for stat, var in vitals_labs_agg.columns]
# ffill imputation
vitals_labs_agg = vitals_labs_agg.sort_index().groupby(level=0).ffill()

vitals_labs_agg = vitals_labs_agg.reset_index()

print(f"\n✓ Aggregated vitals and labs")
print(f"  Shape: {vitals_labs_agg.shape}")
print(f"  Features: {vitals_labs_agg.shape[1] - 2 } (excluding patientid, time_day)")
print(f"  Unique patients: {vitals_labs_agg['patientid'].nunique():,}")
print(f"  Time windows: {len(vitals_labs_agg):,}")


# Drop features with >20% missing values after imputation
missing_pct = vitals_labs_agg.isna().mean()
cols_to_drop = (missing_pct[missing_pct > 0.2].index).tolist()

vitals_labs_clean = vitals_labs_agg.drop(columns=cols_to_drop)

print(f"\n✓ Dropped high-missingness columns (>20%)")
print(f"  Remaining features: {vitals_labs_clean.shape[1] - 2}")
print(f"  Dropped: {vitals_labs_agg.shape[1] - vitals_labs_clean.shape[1]} columns")

# Show which features were kept
feature_cols = [c for c in vitals_labs_clean.columns if c not in ['patientid', 'time_day']]
print(f"\n📊 Kept features ({len(feature_cols)}):")
for col in sorted(feature_cols):
    pct_missing = vitals_labs_clean[col].isna().mean()
    print(f"   {col:40s}: {pct_missing:3.2%} missing")


✓ Calculated time_day for observations
  Time range: 0 to 28 days

📋 Defined valid ranges for 29 variables

✓ Filtered observations
  Before: 155,275,209
  After:  138,579,820
  Removed: 16,695,389 (10.8%)

⏳ Aggregating observations...

✓ Aggregated vitals and labs
  Shape: (31835, 89)
  Features: 87 (excluding patientid, time_day)
  Unique patients: 4,638
  Time windows: 31,835

✓ Dropped high-missingness columns (>20%)
  Remaining features: 63
  Dropped: 24 columns

📊 Kept features (63):
   bicarbonate_max                         : 0.30% missing
   bicarbonate_mean                        : 0.30% missing
   bicarbonate_min                         : 0.30% missing
   bun_max                                 : 4.05% missing
   bun_mean                                : 4.05% missing
   bun_min                                 : 4.05% missing
   calcium_max                             : 0.41% missing
   calcium_mean                            : 0.41% missing
   calcium_min                 

In [19]:
patient_final = patient_final.merge(
    vitals_labs_clean,
    on='patientid',
    how='left'
)

patient_final.to_csv('../../results/hirid/sepsis/patient_features.csv', index=False)

## TRAJECTORY MODELING

Run standalone script for SLURM:
```bash
python ../hirid_sepsis_trajs.py --window-days 3.0 --n-batches 10
```

In [23]:
# Load pre-computed trajectory probabilities
lactate_probs_path = '../../results/hirid/sepsis/lactate_trajectory_probs_bayes.csv'
platelet_probs_path = '../../results/hirid/sepsis/platelet_trajectory_probs_bayes.csv'
wbc_probs_path = '../../results/hirid/sepsis/wbc_trajectory_probs_bayes.csv'

lactate_trajectory_probs = pd.read_csv(lactate_probs_path)
platelet_trajectory_probs = pd.read_csv(platelet_probs_path)
wbc_trajectory_probs = pd.read_csv(wbc_probs_path)

In [24]:
print(lactate_trajectory_probs.head())
print(platelet_trajectory_probs.head())
print(wbc_trajectory_probs.head())

   patientid  time_days  time_day  lactate  baseline_lactate  prob_stable  \
0       3939   0.215972         0      0.9               1.0      0.81625   
1      25928   0.231944         0      3.0               2.8      0.31750   
2       3514   0.257639         0      1.7               1.4      0.16000   
3      11751   0.262500         0      1.3               1.8      0.91375   
4      24090   0.266667         0      1.3               1.5      0.81000   

   prob_gradual_increase  prob_rapid_increase  
0                0.12875                  0.0  
1                0.60625                  0.0  
2                0.72750                  0.0  
3                0.06375                  0.0  
4                0.14625                  0.0  
   patientid  time_days  time_day  platelet  baseline_platelet  prob_stable  \
0      30731   0.379861         0      59.0               99.0      0.17375   
1      32660   0.399306         0     181.0              180.0      0.98000   
2      15613

In [26]:
lactate_trajectory_probs.columns = ['patientid', 'time_day'] + [f'lactate_{col}' for col in lactate_trajectory_probs.columns if col not in ['patientid', 'time_day']]
platelet_trajectory_probs.columns = ['patientid', 'time_day'] + [f'platelet_{col}' for col in platelet_trajectory_probs.columns if col not in ['patientid', 'time_day']]
wbc_trajectory_probs.columns = ['patientid', 'time_day'] + [f'wbc_{col}' for col in wbc_trajectory_probs.columns if col not in ['patientid', 'time_day']]

trajectroy_probs = lactate_trajectory_probs.merge(
    platelet_trajectory_probs, on=['patientid', 'time_day'], how='outer'
).merge(
    wbc_trajectory_probs, on=['patientid', 'time_day'], how='outer'
)

In [ ]:
# # Visualize trajectory distribution
# plot_trajectory_distribution(trajectory_probs)

## SUMMARY FEATURES

In [29]:
LOOKBACK_DAYS = 3
lactate_summary_3d = biomarker_summary_stats(
    lactate_ts.rename(columns={'lab_value': 'lactate'}), 'lactate', lookback_days=LOOKBACK_DAYS
)
platelet_summary_3d = biomarker_summary_stats(
    platelet_ts.rename(columns={'lab_value': 'platelets'}), 'platelets', lookback_days=LOOKBACK_DAYS
)
wbc_summary_3d = biomarker_summary_stats(
    wbc_ts.rename(columns={'lab_value': 'wbc'}), 'wbc', lookback_days=LOOKBACK_DAYS
)

## DEFINE OUTCOME

Septic shock defined as:
1. Hypotension (MAP < 65 mmHg) OR
2. Vasopressor requirement AND
3. Elevated lactate (> 2 mmol/L)

In [33]:
# Configuration
PREDICTION_GAP_DAYS = 0.5  # 12-hour gap
PREDICTION_WINDOW_DAYS = 2.0  # Predict shock within 2 days

print(f"📋 Prediction Configuration:")
print(f"   Gap:    {PREDICTION_GAP_DAYS} days")
print(f"   Window: {PREDICTION_WINDOW_DAYS} days")
print(f"   Total:  {PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS} days lookahead")

# Prepare vasopressor timing
vasopressor_df['charttime'] = pd.to_datetime(vasopressor_df['charttime'])
vasopressor_df['admission_time'] = pd.to_datetime(vasopressor_df['admission_time'])
vasopressor_df['time_days'] = (vasopressor_df['charttime'] - vasopressor_df['admission_time']).dt.total_seconds() / 86400
vasopressor_df['time_day'] = vasopressor_df['time_days'].astype(int)

# Get patients on vasopressors by day
vasopressor_days = vasopressor_df.groupby(['patientid', 'time_day']).size().reset_index(name='vaso_count')
vasopressor_days['on_vasopressors'] = 1

print(f"\n✓ Vasopressor data prepared")
print(f"  Patient-days with vasopressors: {len(vasopressor_days):,}")

📋 Prediction Configuration:
   Gap:    0.5 days
   Window: 2.0 days
   Total:  2.5 days lookahead

✓ Vasopressor data prepared
  Patient-days with vasopressors: 24,080


In [31]:
patient_final.columns

Index(['patientid', 'admission_time', 'sex', 'age', 'discharge_status',
       'n_observations', 'los_days', 'baseline_lactate', 'baseline_platelets',
       'baseline_wbc', 'time_day', 'bicarbonate_min', 'bun_min', 'calcium_min',
       'chloride_min', 'creatinine_min', 'dbp_invasive_min', 'heart_rate_min',
       'hemoglobin_min', 'inr_min', 'lactate_min', 'magnesium_min',
       'mbp_invasive_min', 'phosphate_min', 'platelets_min', 'potassium_min',
       'respiratory_rate_min', 'sbp_invasive_min', 'sodium_min', 'spo2_min',
       'wbc_min', 'weight_min', 'bicarbonate_max', 'bun_max', 'calcium_max',
       'chloride_max', 'creatinine_max', 'dbp_invasive_max', 'heart_rate_max',
       'hemoglobin_max', 'inr_max', 'lactate_max', 'magnesium_max',
       'mbp_invasive_max', 'phosphate_max', 'platelets_max', 'potassium_max',
       'respiratory_rate_max', 'sbp_invasive_max', 'sodium_max', 'spo2_max',
       'wbc_max', 'weight_max', 'bicarbonate_mean', 'bun_mean', 'calcium_mean',
       '

In [36]:
# Define septic shock events (multimarker: lactate + platelets + WBC)
shock_events = []
excluded_counts = {'already_shock': 0, 'no_future_data': 0}


for patientid, grp in patient_final.groupby('patientid'):
    grp = grp.sort_values('time_day')
    baseline_lac = grp['baseline_lactate'].iloc[0] if 'baseline_lactate' in grp.columns else grp['lactate'].iloc[0]
    baseline_plt = grp['baseline_platelets'].iloc[0] if 'baseline_platelets' in grp.columns else np.nan
    baseline_wbc_val = grp['baseline_wbc'].iloc[0] if 'baseline_wbc' in grp.columns else np.nan
    
    for i in range(len(grp)):
        current_time = grp.iloc[i]['time_day']
        current_lactate = grp.iloc[i]['lactate_mean']
        current_platelets = grp.iloc[i].get('platelets_mean', np.nan)
        current_wbc = grp.iloc[i].get('wbc_mean', np.nan)
        
        # Skip if already in shock (lactate > 4)
        if current_lactate > 4.0:
            excluded_counts['already_shock'] += 1
            continue
        
        # Define prediction window
        prediction_start = current_time + PREDICTION_GAP_DAYS + 1
        prediction_end = current_time + PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS + 1
        
        future_window = lactate_ts[
            (lactate_ts['patientid'] == patientid) & 
            (lactate_ts['time_days'].between(prediction_start, prediction_end, inclusive='both'))
        ]
        
        if len(future_window) == 0:
            excluded_counts['no_future_data'] += 1
            continue
        
        # Check for shock: lactate > 2 AND vasopressor use
        high_lactate = (future_window['lab_value'] > 2.0).any()
        very_high_lactate = (future_window['lab_value'] > 4.0).any()
        
        # Check vasopressor use in window
        vaso_in_window = vasopressor_days[
            (vasopressor_days['patientid'] == patientid) &
            (vasopressor_days['time_day'].between(prediction_start, prediction_end, inclusive='both'))
        ]
        
        on_vasopressor = len(vaso_in_window) > 0
        target_shock = int((high_lactate and on_vasopressor) or very_high_lactate)
        
        platelet_fold_change = (
            current_platelets / baseline_plt if pd.notnull(current_platelets) and pd.notnull(baseline_plt) and baseline_plt > 0 else np.nan
        )
        platelet_low = int(pd.notnull(current_platelets) and current_platelets < 150)
        
        wbc_fold_change = (
            current_wbc / baseline_wbc_val if pd.notnull(current_wbc) and pd.notnull(baseline_wbc_val) and baseline_wbc_val > 0 else np.nan
        )
        wbc_high = int(pd.notnull(current_wbc) and current_wbc > 12)
        wbc_low = int(pd.notnull(current_wbc) and current_wbc < 4)
        wbc_abnormal = int(wbc_high or wbc_low)
        
        shock_events.append({
            'patientid': patientid,
            'time_day': current_time,
            'target_septic_shock': target_shock,
            'current_lactate': current_lactate,
            'baseline_lactate': baseline_lac,
            'lactate_fold_change': current_lactate / baseline_lac if baseline_lac > 0 else 1.0,
            'current_platelets': current_platelets,
            'baseline_platelets': baseline_plt,
            'platelet_fold_change': platelet_fold_change,
            'platelet_low': platelet_low,
            'current_wbc': current_wbc,
            'baseline_wbc': baseline_wbc_val,
            'wbc_fold_change': wbc_fold_change,
            'wbc_high': wbc_high,
            'wbc_low': wbc_low,
            'wbc_abnormal': wbc_abnormal,
            'prediction_gap_days': PREDICTION_GAP_DAYS,
            'prediction_window_days': PREDICTION_WINDOW_DAYS
        })

shock_outcomes_df = pd.DataFrame(shock_events)
# keep only first positive event per patient and all negative events
shock_outcomes_df = shock_outcomes_df.sort_values(['patientid', 'time_day'])
positive_first = shock_outcomes_df[shock_outcomes_df['target_septic_shock'] == 1].groupby('patientid').first().reset_index()
all_negatives = shock_outcomes_df[shock_outcomes_df['target_septic_shock'] == 0]
shock_outcomes_df = pd.concat([all_negatives, positive_first], ignore_index=True).sort_values(['patientid', 'time_day']).reset_index(drop=True)



print(f"\n📊 Outcome Definition Summary:")
print(f"   Total prediction windows: {len(shock_outcomes_df):,}")
print(f"   Unique patients: {shock_outcomes_df['patientid'].nunique():,}")
print(f"\n   Septic Shock Events:")
print(f"      Total events: {shock_outcomes_df['target_septic_shock'].sum():,}")
print(f"      Event rate: {100*shock_outcomes_df['target_septic_shock'].mean():.1f}%")
print(f"\n   Platelet data availability: {shock_outcomes_df['current_platelets'].notna().mean()*100:.1f}% of rows")
print(f"   WBC data availability: {shock_outcomes_df['current_wbc'].notna().mean()*100:.1f}% of rows")


📊 Outcome Definition Summary:
   Total prediction windows: 18,914
   Unique patients: 4,230

   Septic Shock Events:
      Total events: 718
      Event rate: 3.8%

   Platelet data availability: 99.2% of rows
   WBC data availability: 99.2% of rows


## CONSTRUCT PREDICTION DATASET

In [37]:
trajectroy_probs.columns

Index(['patientid', 'time_day', 'lactate_time_days', 'lactate_lactate',
       'lactate_baseline_lactate', 'lactate_prob_stable',
       'lactate_prob_gradual_increase', 'lactate_prob_rapid_increase',
       'platelet_time_days', 'platelet_platelet', 'platelet_baseline_platelet',
       'platelet_prob_stable', 'platelet_prob_gradual_increase',
       'platelet_prob_rapid_increase', 'wbc_time_days', 'wbc_wbc',
       'wbc_baseline_wbc', 'wbc_prob_stable', 'wbc_prob_gradual_increase',
       'wbc_prob_rapid_increase'],
      dtype='object')

In [47]:
# Merge trajectory probabilities with outcomes

prediction_dataset = shock_outcomes_df.merge(
        trajectroy_probs[[
            'patientid', 'time_day', 'lactate_prob_stable',
       'lactate_prob_gradual_increase', 'lactate_prob_rapid_increase',
       'platelet_prob_stable', 'platelet_prob_gradual_increase',
       'platelet_prob_rapid_increase', 'wbc_prob_stable',
       'wbc_prob_gradual_increase', 'wbc_prob_rapid_increase'
        ]],
        on=['patientid', 'time_day'],
        how='left'
    )
    
prediction_dataset['lactate_prob_worsening'] = (
        prediction_dataset['lactate_prob_gradual_increase'] + 
        prediction_dataset['lactate_prob_rapid_increase']
    )
prediction_dataset['platelet_prob_worsening'] = (
        prediction_dataset['platelet_prob_gradual_increase'] + 
        prediction_dataset['platelet_prob_rapid_increase']
    )
prediction_dataset['wbc_prob_worsening'] = (
        prediction_dataset['wbc_prob_gradual_increase'] + 
        prediction_dataset['wbc_prob_rapid_increase']
    )
prediction_dataset['multi_marker_worsening'] = (
    prediction_dataset['lactate_prob_worsening'] +
    prediction_dataset['platelet_prob_worsening'] +
    prediction_dataset['wbc_prob_worsening']
) / 3.0

# Merge patient demographics
patient_static = patient_final[['patientid', 'age', 'sex']].drop_duplicates()
prediction_dataset = prediction_dataset.merge(
    patient_static,
    on='patientid',
    how='left'
)

# Add lactate-derived features
prediction_dataset['lactate_elevated'] = (prediction_dataset['current_lactate'] > 2.0).astype(int)
prediction_dataset['platelet_very_low'] = (
    (prediction_dataset['current_platelets'] < 100) & prediction_dataset['current_platelets'].notna()
).astype(int)
prediction_dataset['wbc_high_flag'] = prediction_dataset['wbc_high']
prediction_dataset['wbc_low_flag'] = prediction_dataset['wbc_low']
prediction_dataset['wbc_abnormal_flag'] = prediction_dataset['wbc_abnormal']

prediction_dataset = prediction_dataset.merge(
    lactate_summary_3d.fillna(0),
    on=['patientid', 'time_day'],
    how='left'
).merge(
    platelet_summary_3d.fillna(0),
    on=['patientid', 'time_day'],
    how='left'
).merge(
    wbc_summary_3d.fillna(0),
    on=['patientid', 'time_day'],
    how='left'
)


# Define vitals/labs features list
vitals_labs_features = ['bicarbonate_min', 'bun_min',
       'calcium_min', 'chloride_min', 'creatinine_min', 'dbp_invasive_min',
       'heart_rate_min', 'hemoglobin_min', 'inr_min', 'lactate_min',
       'magnesium_min', 'mbp_invasive_min', 'phosphate_min', 'platelets_min',
       'potassium_min', 'respiratory_rate_min', 'sbp_invasive_min',
       'sodium_min', 'spo2_min', 'wbc_min', 'bicarbonate_max',
       'bun_max', 'calcium_max', 'chloride_max', 'creatinine_max',
       'dbp_invasive_max', 'heart_rate_max', 'hemoglobin_max', 'inr_max',
       'lactate_max', 'magnesium_max', 'mbp_invasive_max', 'phosphate_max',
       'platelets_max', 'potassium_max', 'respiratory_rate_max',
       'sbp_invasive_max', 'sodium_max', 'spo2_max', 'wbc_max',
       'bicarbonate_mean', 'bun_mean', 'calcium_mean', 'chloride_mean',
       'creatinine_mean', 'dbp_invasive_mean', 'heart_rate_mean',
       'hemoglobin_mean', 'inr_mean', 'lactate_mean', 'magnesium_mean',
       'mbp_invasive_mean', 'phosphate_mean', 'platelets_mean',
       'potassium_mean', 'respiratory_rate_mean', 'sbp_invasive_mean',
       'sodium_mean', 'spo2_mean', 'wbc_mean', 'weight_mean']

# Merge vitals and labs features
prediction_dataset = prediction_dataset.merge(
    patient_final[vitals_labs_features + ['patientid', 'time_day']].drop_duplicates(subset=['patientid', 'time_day']),
    on=['patientid', 'time_day'], 
    how='left'
)

# Backfill weight only
prediction_dataset['weight_mean'] = prediction_dataset.groupby('patientid')['weight_mean'].bfill()



print(f"\n📋 Final Prediction Dataset:")
print(f"   Rows: {len(prediction_dataset):,}")
print(f"   Columns: {len(prediction_dataset.columns)}")
print(f"   Unique patients: {prediction_dataset['patientid'].nunique():,}")

# Save
os.makedirs('../../results/hirid/sepsis', exist_ok=True)
prediction_dataset.to_csv('../../results/hirid/sepsis/sepsis_prediction_dataset.csv', index=False)
print(f"\n✓ Saved: results/hirid/sepsis/sepsis_prediction_dataset.csv")


📋 Final Prediction Dataset:
   Rows: 18,914
   Columns: 117
   Unique patients: 4,230

✓ Saved: results/hirid/sepsis/sepsis_prediction_dataset.csv


In [48]:
prediction_dataset['lactate_min_3d'].describe()

count    18359.000000
mean         0.997260
std          0.425374
min          0.300000
25%          0.700000
50%          0.900000
75%          1.200000
max          3.700000
Name: lactate_min_3d, dtype: float64

## FIT MODELS

In [40]:
lactate_summary_3d.columns

Index(['patientid', 'time_day', 'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d'],
      dtype='object')

In [54]:
# Define feature sets
feature_sets = {
    'Lactate Trajectory': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
    ],
    'Multi-marker Trajectory': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
    ],
    'Lactate Summary Stats': ['lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d'],
    'Multi-marker Summary Stats': [
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
    ],
    'Lactate Trajectory + Summary Stats': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
    ],
    'Multi-marker Trajectory + Summary Stats': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
    ],
    
    'Static Only': [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    
    'Lactate Trajectory + Static': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Multimarker Trajectory + Static': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Lactate Summary Stats + Static': [
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Multimarker Summary Stats + Static': [
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',],
        'Lactate Trajectory + Summary Stats + Static': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age','sex'],
    'Multimarker Trajectory + Summary Stats + Static': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Static + Dynamic Vitals/Labs': vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Lactate Trajectory + Static + Dynamic Vitals/Labs': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Multimarker Trajectory + Static + Dynamic Vitals/Labs': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Lactate Sammary Stats + Static + Dynamic Vitals/Labs': [
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Multi-marker Summary Stats + Static + Dynamic Vitals/Labs': [
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
    'Lactate Trajectory + Summary Stats + Static + Dynamic Vitals/Labs': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age', 
        'sex'],
    'Multimarker Trajectory + Summary Stats + Static + Dynamic Vitals/Labs': [
        'lactate_prob_stable',
        'lactate_prob_gradual_increase',
        'lactate_prob_rapid_increase',
        'lactate_prob_worsening',
        'platelet_prob_stable',
        'platelet_prob_gradual_increase',
        'platelet_prob_rapid_increase',
        'platelet_prob_worsening',
        'wbc_prob_stable',
        'wbc_prob_gradual_increase',
        'wbc_prob_rapid_increase',
        'wbc_prob_worsening',
        'multi_marker_worsening',
        'lactate_mean_3d', 'lactate_max_3d',
       'lactate_min_3d', 'lactate_change_3d', 'lactate_linear_trend_3d',
       'lactate_std_3d',
       'platelets_mean_3d', 'platelets_max_3d', 'platelets_min_3d',
       'platelets_change_3d', 'platelets_linear_trend_3d', 'platelets_std_3d',
       'wbc_mean_3d', 'wbc_max_3d', 'wbc_min_3d', 'wbc_change_3d',
       'wbc_linear_trend_3d', 'wbc_std_3d',
    ] + vitals_labs_features + [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
        'sex',
    ],
}

print(f"📋 Feature Set Summary:")
for name, features in feature_sets.items():
    print(f"   {name:35s}: {len(features):2d} features")

📋 Feature Set Summary:
   Lactate Trajectory                 :  4 features
   Multi-marker Trajectory            : 13 features
   Lactate Summary Stats              :  6 features
   Multi-marker Summary Stats         : 18 features
   Lactate Trajectory + Summary Stats : 10 features
   Multi-marker Trajectory + Summary Stats: 31 features
   Static Only                        : 14 features
   Lactate Trajectory + Static        : 18 features
   Multimarker Trajectory + Static    : 27 features
   Lactate Summary Stats + Static     : 20 features
   Multimarker Summary Stats + Static : 32 features
   Lactate Trajectory + Summary Stats + Static: 24 features
   Multimarker Trajectory + Summary Stats + Static: 45 features
   Static + Dynamic Vitals/Labs       : 75 features
   Lactate Trajectory + Static + Dynamic Vitals/Labs: 79 features
   Multimarker Trajectory + Static + Dynamic Vitals/Labs: 88 features
   Lactate Sammary Stats + Static + Dynamic Vitals/Labs: 81 features
   Multi-marker Summ

In [50]:
# Prepare data for modeling
prediction_clean = prediction_dataset.dropna(subset=['target_septic_shock'])

if 'sex' in prediction_clean.columns:
    prediction_clean['sex'] = prediction_clean['sex'].map({'m': 1, 'f': 0, 'M': 1, 'F': 0})

for col in prediction_clean.columns:
    if col in feature_sets['Multi-marker Trajectory']:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=2)
        prediction_clean[col] = prediction_clean[col].fillna(0)

y = prediction_clean['target_septic_shock']
groups = prediction_clean['patientid']

print(f"\n📋 Final Dataset for Modeling:")
print(f"   Total samples: {len(y):,}")
print(f"   Positive class: {y.sum():,} ({100*y.mean():.1f}%)")
print(f"   Unique patients: {groups.nunique():,}")


📋 Final Dataset for Modeling:
   Total samples: 18,914
   Positive class: 718 (3.8%)
   Unique patients: 4,230


## EVALUATE

In [55]:
# Repeated Cross-validation
n_repeats = 10
n_folds = 5
n_folds_total = n_repeats * n_folds

comparison_results = {}

print(f"\n🎯 Training models with {n_repeats}-Repeat {n_folds}-Fold CV:\n")

for feature_set_name, feature_cols in feature_sets.items():
    print(f"{feature_set_name}")
    print("-" * 60)
    
    fold_metrics = train_repeated_cv(
        prediction_df=prediction_clean,
        feature_cols=feature_cols,
        target_col='target_septic_shock',
        group_col='patientid',
        n_repeats=n_repeats,
        n_folds=n_folds,
        random_state=920
    )
    
    comparison_results[feature_set_name] = fold_metrics
    
    print(f"  AUROC:   {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}")
    print(f"  AUPR:    {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}\n")


🎯 Training models with 10-Repeat 5-Fold CV:

Lactate Trajectory
------------------------------------------------------------
  AUROC:   0.500 ± 0.000
  AUPR:    0.038 ± 0.003

Multi-marker Trajectory
------------------------------------------------------------
  AUROC:   0.500 ± 0.000
  AUPR:    0.038 ± 0.003

Lactate Summary Stats
------------------------------------------------------------
  AUROC:   0.776 ± 0.026
  AUPR:    0.190 ± 0.041

Multi-marker Summary Stats
------------------------------------------------------------
  AUROC:   0.791 ± 0.015
  AUPR:    0.210 ± 0.034

Lactate Trajectory + Summary Stats
------------------------------------------------------------
  AUROC:   0.776 ± 0.026
  AUPR:    0.190 ± 0.041

Multi-marker Trajectory + Summary Stats
------------------------------------------------------------
  AUROC:   0.791 ± 0.015
  AUPR:    0.210 ± 0.034

Static Only
------------------------------------------------------------
  AUROC:   0.773 ± 0.009
  AUPR:    0.156 

In [ ]:
# Performance comparison
summary_df = pd.DataFrame([
    {
        'Model': name,
        'ROC-AUC': f"{np.mean(metrics['roc_auc']):.3f} ± {np.std(metrics['roc_auc']):.3f}",
        'Avg Precision': f"{np.mean(metrics['avg_precision']):.3f} ± {np.std(metrics['avg_precision']):.3f}"
    }
    for name, metrics in comparison_results.items()
])

print("\n📊 Performance Summary:")
print(summary_df.to_string(index=False))

# ROC & PR Curves
fig, (ax1, ax2) = plot_roc_pr_curves(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    color_scheme='Set1'
)
plt.savefig('../../results/hirid/sepsis/model_comparison_curves.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_curves.png")
plt.show()

In [ ]:
# Statistical significance testing
pairs_to_compare = [(0, 1), (1, 2)]

fig, (ax1, ax2) = plot_boxplots_with_stats(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    pairs_to_compare=pairs_to_compare,
    n_folds_total=n_folds_total
)
plt.savefig('../../results/hirid/sepsis/model_comparison_boxplots.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_boxplots.png")
plt.show()

print_statistical_comparisons(
    comparison_results=comparison_results,
    pairs_to_compare=pairs_to_compare
)